# Return Autocorrelation Analysis: BTC vs SOL

**目标**: 在不同时间尺度上分析收益率的自回归结构，确定：
- 哪些周期存在 **正自相关**（动量效应）
- 哪些周期存在 **负自相关**（均值回复）
- 从而确定模型的最佳预测目标

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, timedelta
from utilities.binance_loader import load_agg_trades, resample_trades_to_ohlcv

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['font.size'] = 11

## 1. Load Data

In [ ]:
# BTC: Sep-Oct 2022 (~20 days cached)
btc_dates = [
    '2022-09-05','2022-09-06','2022-09-07','2022-09-08',
    '2022-10-01','2022-10-02','2022-10-03','2022-10-04','2022-10-05',
    '2022-10-06','2022-10-07','2022-10-08','2022-10-09','2022-10-10',
    '2022-10-11','2022-10-13','2022-10-14','2022-10-15','2022-10-16',
]

# SOL: Jan-Feb 2026 (10 days cached)
sol_dates = [
    '2026-01-28','2026-01-29','2026-01-30','2026-01-31',
    '2026-02-01','2026-02-02','2026-02-03','2026-02-04',
    '2026-02-05','2026-02-06',
]

def load_trades_multi(symbol, dates):
    dfs = []
    for d in dates:
        dt = date.fromisoformat(d)
        t = load_agg_trades(symbol, dt, dt + timedelta(days=1))
        if not t.empty:
            dfs.append(t)
            print(f"  {d}: {len(t):,} trades")
    return pd.concat(dfs).sort_index() if dfs else pd.DataFrame()

print('BTC/USDT:')
btc_trades = load_trades_multi('BTC/USDT', btc_dates)
print(f'  Total: {len(btc_trades):,} trades')

print('\nSOL/USDT:')
sol_trades = load_trades_multi('SOL/USDT', sol_dates)
print(f'  Total: {len(sol_trades):,} trades')

## 2. Resample to Multiple Frequencies

In [ ]:
# 多频率 OHLCV
freqs = ['1s', '5s', '10s', '30s', '1min', '5min', '15min', '30min', '1h']

btc_ohlcv = {}
sol_ohlcv = {}

for f in freqs:
    btc_ohlcv[f] = resample_trades_to_ohlcv(btc_trades, f)
    sol_ohlcv[f] = resample_trades_to_ohlcv(sol_trades, f)
    print(f'{f:>5}: BTC {len(btc_ohlcv[f]):>7,} bars  |  SOL {len(sol_ohlcv[f]):>7,} bars')

## 3. Compute Return Autocorrelation at Each Frequency

对于每个频率，计算 `log_return` 的 autocorrelation 在 lag 1~50 上的值。  
- **正值** = 动量（当前涨 → 下一期倾向于继续涨）
- **负值** = 均值回复（当前涨 → 下一期倾向于跌回来）

In [ ]:
def compute_autocorr(ohlcv_dict, max_lag=50):
    """计算各频率的收益率自相关函数。"""
    results = {}
    for freq, ohlcv in ohlcv_dict.items():
        close = ohlcv['close'].astype(float)
        ret = np.log(close / close.shift(1)).dropna()
        
        # 按天去均值（去除日内趋势）
        if hasattr(ret.index, 'date'):
            daily_mean = ret.groupby(ret.index.date).transform('mean')
            ret_demean = ret - daily_mean
        else:
            ret_demean = ret - ret.mean()
        
        # 计算 lag 1 到 max_lag 的自相关
        actual_max_lag = min(max_lag, len(ret_demean) // 10)
        acf = [ret_demean.autocorr(lag=k) for k in range(1, actual_max_lag + 1)]
        results[freq] = {
            'acf': acf,
            'lags': list(range(1, actual_max_lag + 1)),
            'n': len(ret_demean),
            'ret_std': float(ret.std()),
            'ret_mean': float(ret.mean()),
        }
    return results

btc_acf = compute_autocorr(btc_ohlcv, max_lag=50)
sol_acf = compute_autocorr(sol_ohlcv, max_lag=50)

print('Done. Return stats (log return):')
print(f'{"Freq":>5} | {"BTC mean":>10} {"BTC std":>10} {"BTC N":>8} | {"SOL mean":>10} {"SOL std":>10} {"SOL N":>8}')
print('-' * 80)
for f in freqs:
    b = btc_acf[f]
    s = sol_acf[f]
    print(f'{f:>5} | {b["ret_mean"]:>10.6f} {b["ret_std"]:>10.6f} {b["n"]:>8,} | {s["ret_mean"]:>10.6f} {s["ret_std"]:>10.6f} {s["n"]:>8,}')

## 4. Autocorrelation at Lag 1 Across Frequencies

最关键的一张图：lag-1 自相关 vs. 采样频率。  
- **负值** = 该频率下 return 存在反转（均值回复）  
- **正值** = 该频率下 return 存在动量  
- 接近 0 = 该频率下 return 近似随机游走

In [ ]:
# Lag-1 autocorrelation across frequencies
fig, ax = plt.subplots(figsize=(14, 6))

freq_labels = freqs
btc_lag1 = [btc_acf[f]['acf'][0] for f in freqs]
sol_lag1 = [sol_acf[f]['acf'][0] for f in freqs]

x = np.arange(len(freqs))
w = 0.35
bars1 = ax.bar(x - w/2, btc_lag1, w, label='BTC/USDT', color='#FF9800', alpha=0.85)
bars2 = ax.bar(x + w/2, sol_lag1, w, label='SOL/USDT', color='#2196F3', alpha=0.85)

ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(freq_labels, fontsize=12)
ax.set_ylabel('Lag-1 Autocorrelation', fontsize=13)
ax.set_xlabel('Bar Frequency', fontsize=13)
ax.set_title('Return Lag-1 Autocorrelation by Frequency\n(Negative = Mean Reversion, Positive = Momentum)', fontsize=14)
ax.legend(fontsize=12)

# 2-sigma confidence band
for f_idx, f in enumerate(freqs):
    n_btc = btc_acf[f]['n']
    n_sol = sol_acf[f]['n']
    ci_btc = 2 / np.sqrt(n_btc)
    ci_sol = 2 / np.sqrt(n_sol)
    # Show significance markers
    if abs(btc_lag1[f_idx]) > ci_btc:
        ax.annotate('*', (f_idx - w/2, btc_lag1[f_idx]), ha='center', va='bottom' if btc_lag1[f_idx] > 0 else 'top', fontsize=16, fontweight='bold', color='red')
    if abs(sol_lag1[f_idx]) > ci_sol:
        ax.annotate('*', (f_idx + w/2, sol_lag1[f_idx]), ha='center', va='bottom' if sol_lag1[f_idx] > 0 else 'top', fontsize=16, fontweight='bold', color='blue')

ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../notebooks/lag1_autocorr_by_freq.png', dpi=150)
plt.show()

# Print values
print('\nLag-1 Autocorrelation (* = significant at 95% CI):')
print(f'{"Freq":>5} | {"BTC":>8} {"sig?":>5} | {"SOL":>8} {"sig?":>5}')
print('-' * 42)
for i, f in enumerate(freqs):
    ci_b = 2 / np.sqrt(btc_acf[f]['n'])
    ci_s = 2 / np.sqrt(sol_acf[f]['n'])
    sig_b = '***' if abs(btc_lag1[i]) > ci_b else ''
    sig_s = '***' if abs(sol_lag1[i]) > ci_s else ''
    print(f'{f:>5} | {btc_lag1[i]:>+8.4f} {sig_b:>5} | {sol_lag1[i]:>+8.4f} {sig_s:>5}')

## 5. Full ACF Plots (Lag 1~50) at Key Frequencies

In [ ]:
key_freqs = ['1s', '5s', '10s', '30s', '1min', '5min']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Return Autocorrelation Function (ACF)\nBTC (orange) vs SOL (blue)', fontsize=15, y=1.02)

for idx, f in enumerate(key_freqs):
    ax = axes[idx // 3, idx % 3]
    
    b_acf = btc_acf[f]['acf']
    s_acf = sol_acf[f]['acf']
    b_lags = btc_acf[f]['lags']
    s_lags = sol_acf[f]['lags']
    
    # Confidence intervals
    ci_b = 2 / np.sqrt(btc_acf[f]['n'])
    ci_s = 2 / np.sqrt(sol_acf[f]['n'])
    
    ax.bar(np.array(b_lags) - 0.2, b_acf, width=0.4, color='#FF9800', alpha=0.7, label='BTC')
    ax.bar(np.array(s_lags[:len(s_acf)]) + 0.2, s_acf, width=0.4, color='#2196F3', alpha=0.7, label='SOL')
    
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.axhline(y=ci_b, color='#FF9800', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.axhline(y=-ci_b, color='#FF9800', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.axhline(y=ci_s, color='#2196F3', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.axhline(y=-ci_s, color='#2196F3', linewidth=0.8, linestyle='--', alpha=0.5)
    
    ax.set_title(f'{f} bars', fontsize=13, fontweight='bold')
    ax.set_xlabel('Lag')
    ax.set_ylabel('ACF')
    if idx == 0:
        ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../notebooks/acf_by_freq.png', dpi=150)
plt.show()

## 6. Lag-1 Autocorrelation Heatmap (Per-Day Stability)

检验自相关是否在不同天稳定存在，还是只是个别天的异常。

In [ ]:
def daily_lag1_acf(trades, symbol_name, dates, freqs_subset):
    """按天计算各频率的 lag-1 autocorrelation。"""
    results = []
    for d in dates:
        dt = date.fromisoformat(d)
        t = load_agg_trades(symbol_name, dt, dt + timedelta(days=1))
        if t.empty:
            continue
        for f in freqs_subset:
            ohlcv = resample_trades_to_ohlcv(t, f)
            close = ohlcv['close'].astype(float)
            ret = np.log(close / close.shift(1)).dropna()
            ret = ret - ret.mean()  # demean
            if len(ret) > 20:
                acf1 = ret.autocorr(lag=1)
                results.append({'date': d, 'freq': f, 'acf1': acf1, 'n': len(ret)})
    return pd.DataFrame(results)

check_freqs = ['1s', '5s', '10s', '30s', '1min', '5min']

print('Computing daily lag-1 ACF for BTC...')
btc_daily = daily_lag1_acf(btc_trades, 'BTC/USDT', btc_dates, check_freqs)
print('Computing daily lag-1 ACF for SOL...')
sol_daily = daily_lag1_acf(sol_trades, 'SOL/USDT', sol_dates, check_freqs)

# Pivot for heatmap
btc_pivot = btc_daily.pivot(index='date', columns='freq', values='acf1')[check_freqs]
sol_pivot = sol_daily.pivot(index='date', columns='freq', values='acf1')[check_freqs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, max(6, len(btc_dates) * 0.4)))

# BTC heatmap
im1 = ax1.imshow(btc_pivot.values, cmap='RdBu_r', aspect='auto', vmin=-0.15, vmax=0.15)
ax1.set_xticks(range(len(check_freqs)))
ax1.set_xticklabels(check_freqs)
ax1.set_yticks(range(len(btc_pivot.index)))
ax1.set_yticklabels(btc_pivot.index, fontsize=9)
ax1.set_title('BTC/USDT - Daily Lag-1 ACF', fontsize=13)
for i in range(btc_pivot.shape[0]):
    for j in range(btc_pivot.shape[1]):
        v = btc_pivot.values[i, j]
        if not np.isnan(v):
            ax1.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=8)

# SOL heatmap
im2 = ax2.imshow(sol_pivot.values, cmap='RdBu_r', aspect='auto', vmin=-0.15, vmax=0.15)
ax2.set_xticks(range(len(check_freqs)))
ax2.set_xticklabels(check_freqs)
ax2.set_yticks(range(len(sol_pivot.index)))
ax2.set_yticklabels(sol_pivot.index, fontsize=9)
ax2.set_title('SOL/USDT - Daily Lag-1 ACF', fontsize=13)
for i in range(sol_pivot.shape[0]):
    for j in range(sol_pivot.shape[1]):
        v = sol_pivot.values[i, j]
        if not np.isnan(v):
            ax2.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=8)

fig.colorbar(im1, ax=[ax1, ax2], label='Lag-1 ACF', shrink=0.8)
plt.suptitle('Daily Lag-1 Autocorrelation Stability\n(Red=Momentum, Blue=Mean Reversion)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../notebooks/daily_acf_heatmap.png', dpi=150)
plt.show()

## 7. Summary Statistics

In [ ]:
# Mean and std of daily lag-1 ACF across days
print('='*70)
print('SUMMARY: Mean Lag-1 ACF Across Days (± std)')
print('='*70)
print(f'{"Freq":>5} | {"BTC mean":>9} {"BTC std":>9} {"BTC t-stat":>10} | {"SOL mean":>9} {"SOL std":>9} {"SOL t-stat":>10}')
print('-'*80)
for f in check_freqs:
    btc_vals = btc_daily[btc_daily['freq'] == f]['acf1'].dropna()
    sol_vals = sol_daily[sol_daily['freq'] == f]['acf1'].dropna()
    
    b_mean = btc_vals.mean()
    b_std = btc_vals.std()
    b_t = b_mean / (b_std / np.sqrt(len(btc_vals)) + 1e-10) if len(btc_vals) > 1 else 0
    
    s_mean = sol_vals.mean()
    s_std = sol_vals.std()
    s_t = s_mean / (s_std / np.sqrt(len(sol_vals)) + 1e-10) if len(sol_vals) > 1 else 0
    
    b_sig = '***' if abs(b_t) > 2.5 else '**' if abs(b_t) > 2 else '*' if abs(b_t) > 1.5 else ''
    s_sig = '***' if abs(s_t) > 2.5 else '**' if abs(s_t) > 2 else '*' if abs(s_t) > 1.5 else ''
    
    print(f'{f:>5} | {b_mean:>+9.4f} {b_std:>9.4f} {b_t:>+8.2f} {b_sig:<2} | {s_mean:>+9.4f} {s_std:>9.4f} {s_t:>+8.2f} {s_sig:<2}')

print()
print('Interpretation:')
print('  * p<0.10  ** p<0.05  *** p<0.01')
print('  Negative ACF = Mean Reversion (predictable reversal)')
print('  Positive ACF = Momentum (predictable continuation)')

## 8. Multi-Lag Signed ACF Decay Plot

观察 ACF 从 lag-1 到 lag-50 的衰减模式，寻找持续性信号。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (name, acf_dict) in zip(axes, [('BTC/USDT', btc_acf), ('SOL/USDT', sol_acf)]):
    for f in ['1s', '5s', '10s', '30s', '1min', '5min']:
        acf = acf_dict[f]['acf']
        lags = acf_dict[f]['lags']
        ax.plot(lags[:30], acf[:30], marker='.', markersize=4, label=f, alpha=0.8)
    
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_xlabel('Lag', fontsize=12)
    ax.set_ylabel('ACF', fontsize=12)
    ax.set_title(f'{name} - ACF Decay by Frequency', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../notebooks/acf_decay.png', dpi=150)
plt.show()

## 9. Cumulative Return if Trading on Lag-1 Signal

简单验证：如果 ACF < 0（均值回复），做 contrarian 策略（上一期涨→做空，上一期跌→做多），扣除成本后是否盈利？

In [ ]:
def simple_contrarian_pnl(ohlcv, cost_bps=4):
    """简单均值回复策略: signal = -sign(last_return), 每 bar 交易。"""
    close = ohlcv['close'].astype(float)
    ret = close.pct_change().dropna() * 10000  # bps
    signal = -np.sign(ret.shift(1))  # contrarian
    gross_pnl = signal * ret  # bps
    net_pnl = gross_pnl - cost_bps * 2  # round-trip cost every bar
    return pd.DataFrame({
        'gross_pnl': gross_pnl,
        'net_pnl': net_pnl,
        'gross_cum': gross_pnl.cumsum(),
        'net_cum': net_pnl.cumsum(),
    }).dropna()

# Test at key frequencies
test_freqs = ['1s', '5s', '10s', '30s', '1min', '5min']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Naive Contrarian Strategy (Mean Reversion Test)\nGross (dashed) vs Net of 8bps RT cost (solid)', fontsize=14, y=1.02)

for idx, f in enumerate(test_freqs):
    ax = axes[idx // 3, idx % 3]
    
    btc_pnl = simple_contrarian_pnl(btc_ohlcv[f], cost_bps=4)
    sol_pnl = simple_contrarian_pnl(sol_ohlcv[f], cost_bps=4)
    
    ax.plot(btc_pnl['gross_cum'].values, '--', color='#FF9800', alpha=0.5, label='BTC gross')
    ax.plot(btc_pnl['net_cum'].values, '-', color='#FF9800', alpha=0.9, label='BTC net')
    ax.plot(sol_pnl['gross_cum'].values, '--', color='#2196F3', alpha=0.5, label='SOL gross')
    ax.plot(sol_pnl['net_cum'].values, '-', color='#2196F3', alpha=0.9, label='SOL net')
    
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_title(f'{f} bars', fontsize=13, fontweight='bold')
    ax.set_ylabel('Cum PnL (bps)')
    if idx == 0:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../notebooks/contrarian_pnl.png', dpi=150)
plt.show()

# Summary
print('\nContrarian Strategy Summary (net of 8bps RT cost):')
print(f'{"Freq":>5} | {"BTC gross":>10} {"BTC net":>10} {"BTC Sharpe":>10} | {"SOL gross":>10} {"SOL net":>10} {"SOL Sharpe":>10}')
print('-'*80)
for f in test_freqs:
    bp = simple_contrarian_pnl(btc_ohlcv[f], cost_bps=4)
    sp = simple_contrarian_pnl(sol_ohlcv[f], cost_bps=4)
    
    b_sharpe = bp['net_pnl'].mean() / (bp['net_pnl'].std() + 1e-10) * np.sqrt(len(bp))
    s_sharpe = sp['net_pnl'].mean() / (sp['net_pnl'].std() + 1e-10) * np.sqrt(len(sp))
    
    print(f'{f:>5} | {bp["gross_cum"].iloc[-1]:>+10.1f} {bp["net_cum"].iloc[-1]:>+10.1f} {b_sharpe:>+10.2f} | {sp["gross_cum"].iloc[-1]:>+10.1f} {sp["net_cum"].iloc[-1]:>+10.1f} {s_sharpe:>+10.2f}')

## 10. Conclusion & Prediction Target Recommendation

基于以上分析，确定最佳预测目标周期。

In [ ]:
print('='*60)
print('FINDINGS')
print('='*60)
print()
print('1. Lag-1 ACF Pattern:')
for f in check_freqs:
    b = btc_acf[f]['acf'][0]
    s = sol_acf[f]['acf'][0]
    b_type = 'MOMENTUM' if b > 0.01 else 'REVERSION' if b < -0.01 else 'RANDOM'
    s_type = 'MOMENTUM' if s > 0.01 else 'REVERSION' if s < -0.01 else 'RANDOM'
    print(f'   {f:>5}: BTC={b:+.4f} ({b_type:>9})  SOL={s:+.4f} ({s_type:>9})')

print()
print('2. Recommendation:')
print('   - Frequencies with strong negative ACF → mean reversion target')
print('   - Frequencies with strong positive ACF → momentum target')
print('   - Choose the frequency where |ACF| is largest AND stable across days')